## Overall workflow

* Load Data -> Define model -> Set loss & optimizer -> Train -> Evaluate

In [1]:
# Setup

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Chooses where computation runs.
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [2]:
# Load Data

transform = transforms.ToTensor()

# train=True/False --> True loads 60,000 training images, False loads 10,000
train_data = datasets.MNIST(root="../data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="../data", train=False, download=True, transform=transform)

# batch_size=64 --> Feeds 64 images at a time instead of all at once, which saves memory
# and stabilizes training
# shuffle=True/False --> Training data is shuffled each epoch so the model can't learn the order.
# Test data only needs scoring, so no shuffling is needed.
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

images, labels = next(iter(train_loader))
print(images.shape, labels.shape)
print(labels[:10])

100.0%
100.0%
100.0%
100.0%

torch.Size([64, 1, 28, 28]) torch.Size([64])
tensor([0, 7, 6, 6, 7, 3, 5, 3, 3, 3])


In [3]:
# Define the Model

class MLP(nn.Module):
	def __init__(self):
		super().__init__()
		self.net = nn.Sequential(
			nn.Flatten(),   # Flattens each 28x28 image into a 794-length vector
			nn.Linear(28 * 28, 128),   # Maps 784 inputs to 128 values. Holds learnable weights
			nn.ReLU(),   # Sets negative values to zero
			nn.Linear(128, 10),   # Outputs 10 scores, one for each digit 0-9
		)

	def forward(self, x):
		return self.net(x)

model = MLP().to(device)
print(model)

MLP(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [4]:
# Loss Function and Optimizer

# Measures how far predictions are from the true labels
loss_fn = nn.CrossEntropyLoss()
# Updates the weights in the direction that reduces the loss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [5]:
# Training and Evaluation Functions

# Runs one full pass over the training data
def train_one_epoch(model, loader):
	model.train()
	total_loss = 0
	for images, labels in loader:
		images, labels = images.to(device), labels.to(device)

		preds = model(images)	# 1. Predict
		loss = loss_fn(preds, labels)	# 2. Compute loss

		optimizer.zero_grad()	# 3. Reset gradients
		loss.backward()		# 4. Compute gradients
		optimizer.step()	 # 5. Update weights

		total_loss += loss.item()
	return total_loss / len(loader)

# Measures accuracy on the test data
def evaluate(model, loader):
	model.eval()
	correct = 0
	with torch.no_grad():
		for images, labels in loader:
			images, labels = images.to(device), labels.to(device)
			preds = model(images)
			correct += (preds.argmax(dim=1) == labels).sum().item()
	return correct / len(loader.dataset)

In [7]:
# Run

# Loss should decrease and test acc should rise each epoch
# The target is test_acc >= 0.97
for epoch in range(5):
	loss = train_one_epoch(model, train_loader)
	acc = evaluate(model, test_loader)
	print(f"Epoch {epoch + 1}: loss={loss:.4f}, test_acc={acc:.4f}")

Epoch 1: loss=0.1553, test_acc=0.9625
Epoch 2: loss=0.1095, test_acc=0.9690
Epoch 3: loss=0.0830, test_acc=0.9735
Epoch 4: loss=0.0666, test_acc=0.9767
Epoch 5: loss=0.0527, test_acc=0.9770


## Experiments

In [10]:
### Experiments

def run_experiment(lr, epochs=5):
	torch.manual_seed(42)
	global optimizer
	model = MLP().to(device)	# New model
	optimizer = torch.optim.Adam(model.parameters(), lr=lr)		# New optimizer
	for epoch in range(epochs):
		loss = train_one_epoch(model, train_loader)
		acc = evaluate(model, test_loader)
		print(f"lr={lr} | Epoch {epoch + 1}: loss={loss:.4f}, test_acc={acc:.4f}")
	return acc

### Experiment 1: Learning rate

In [12]:
for lr in [1e-1, 1e-3, 1e-5]:
	run_experiment(lr)

lr=0.1 | Epoch 1: loss=0.9476, test_acc=0.8188
lr=0.1 | Epoch 2: loss=0.8800, test_acc=0.6843
lr=0.1 | Epoch 3: loss=0.9083, test_acc=0.7481
lr=0.1 | Epoch 4: loss=0.9263, test_acc=0.7376
lr=0.1 | Epoch 5: loss=0.9914, test_acc=0.6946
lr=0.001 | Epoch 1: loss=0.3447, test_acc=0.9434
lr=0.001 | Epoch 2: loss=0.1585, test_acc=0.9606
lr=0.001 | Epoch 3: loss=0.1090, test_acc=0.9648
lr=0.001 | Epoch 4: loss=0.0830, test_acc=0.9725
lr=0.001 | Epoch 5: loss=0.0644, test_acc=0.9755
lr=1e-05 | Epoch 1: loss=2.0427, test_acc=0.7200
lr=1e-05 | Epoch 2: loss=1.4965, test_acc=0.7986
lr=1e-05 | Epoch 3: loss=1.1002, test_acc=0.8389
lr=1e-05 | Epoch 4: loss=0.8556, test_acc=0.8560
lr=1e-05 | Epoch 5: loss=0.7051, test_acc=0.8694
